In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
employees = [(1, "Scott", "Tiger", 1000.0, 10,
                      "united states", "+1 123 456 7890", "123 45 6789"
                     ),
                     (2, "Henry", "Ford", 1250.0, None,
                      "India", "+91 234 567 8901", "456 78 9123"
                     ),
                     (3, "Nick", "Junior", 750.0, '',
                      "united KINGDOM", "+44 111 111 1111", "222 33 4444"
                     ),
                     (4, "Bill", "Gomes", 1500.0, 10,
                      "AUSTRALIA", "+61 987 654 3210", "789 12 6118"
                     )
            ]

In [ ]:
df_employees = spark. \
    createDataFrame(employees,
                    schema="""employee_id INT, first_name STRING, 
                    last_name STRING, salary FLOAT, bonus STRING, nationality STRING,
                    phone_number STRING, ssn STRING"""
                   )

In [ ]:
df_employees.show()

In [ ]:
#При работе с null нужно уметь использовать coalesce
#Как мы видим, кроме null у нас в колонке bonus есть пробел - "". В этом случае coalesce не поможет

from pyspark.sql.functions import coalesce, lit

df_employees. \
    withColumn('bonus1', coalesce('bonus', lit(0))). \
    show()

help(coalesce)

In [ ]:
#Приведем тип данных к int, так мы избавились от пробела ""
from pyspark.sql.functions import col

df_employees. \
    withColumn('bonus1', col('bonus').cast('int')). \
    show()

In [ ]:
# Если совместить coalesce и приведение к int, получим правильный вариант колонки bonus1
df_employees. \
    withColumn('bonus1', coalesce(col('bonus').cast('int'), lit(0))). \
    show()

In [ ]:
#Посчитаем колонку payment с использованием coalesce
df_employees. \
    withColumn('payment', col('salary') + (col('salary') * coalesce(col('bonus').cast('int'), lit(0)) / 100)). \
    show()

In [ ]:
spark.stop()